# 🏦 Dossiê de Inteligência FNE v6.0
### *Solução de Auditoria e Elegibilidade — Jornada CODERS*

Este sistema consolida a análise técnica de bens para financiamento, integrando **BrasilAPI** (NCM) e **BNDES** (Catálogo CFI).

**Novidades da v6.0:**
* 📊 **Aba de Comparação:** Adicione múltiplos itens e compare lado a lado.
* 📄 **PDF Reformulado:** Layout formal com cabeçalho, tabelas e rodapé profissionais.
* 🔁 **Token Automático:** Renovação transparente sem quebrar a sessão.
* 🧹 **Código Limpo:** Arquitetura modular para fácil manutenção.

## 🔐 Bloco 1 — Segurança e Autenticação
*Gerenciamento de token OAuth2 com renovação automática ao expirar.*

In [1]:
# =============================================================================
# BLOCO 1 — IMPORTAÇÕES E AUTENTICAÇÃO OAUTH2
# =============================================================================
# Responsável por:
#   1. Importar todas as bibliotecas utilizadas no projeto
#   2. Gerenciar credenciais e tokens de acesso à API do BNDES
#
# Fluxo de autenticação (OAuth2 Client Credentials):
#   App → envia KEY + SECRET → Gateway BNDES → retorna Access Token
#   O token expira em ~1 hora. A classe BNDESAuth renova automaticamente.
# =============================================================================

# Instala as bibliotecas necessárias para o projeto
!pip install fpdf2 > /dev/null 2>&1 # Silencia a saída para não poluir o notebook
!pip install playwright nest_asyncio > /dev/null 2>&1 # Instala Playwright e nest_asyncio
!playwright install chromium > /dev/null 2>&1 # Instala o navegador Chromium para Playwright

# ── Bibliotecas padrão do Python ─────────────────────────────────────────────
import re                          # Expressões regulares (limpeza de CPF, CNPJ, NCM)
import json                        # Serialização de dados para exportação do CHANGELOG
import time                        # Pausas durante captura de screenshot web
import os                          # Operações de arquivo (remoção de temporários)
from datetime import datetime      # Data e hora para logs e controle de expiração do token

# ── Bibliotecas de terceiros ──────────────────────────────────────────────────
import requests                    # Requisições HTTP para BrasilAPI e BNDES
import urllib3                     # Supressão de avisos SSL em redes corporativas
import pytz                        # Fuso horário de Brasília para timestamps precisos
import ipywidgets as widgets        # Componentes visuais interativos no Colab
from IPython.display import display, HTML   # Renderização de HTML dentro do notebook
from fpdf import FPDF, XPos, YPos              # Geração do relatório PDF
from PIL import Image              # Processamento de imagens para o anexo web no PDF
from google.colab import files     # Download de arquivos para o computador do usuário

# ── Playwright + nest_asyncio (captura de screenshots no Colab) ──────────────
# O Colab executa dentro de um loop asyncio. O Playwright sync_api não funciona
# nesse contexto. A solução é:
#   1. Usar a API assíncrona do Playwright (async_api)
#   2. nest_asyncio permite rodar asyncio.run() dentro do loop já existente
import asyncio
import nest_asyncio
nest_asyncio.apply()           # Habilita loops aninhados no Colab
from playwright.async_api import async_playwright

# Suprime avisos de certificado SSL inválido (comum em ambientes corporativos/BNDES)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


# =============================================================================
# CLASSE: BNDESAuth
# =============================================================================
class BNDESAuth:
    """
    Gerencia a autenticação OAuth2 com o Gateway de APIs do BNDES.

    Implementa o fluxo 'Client Credentials', onde a aplicação se autentica
    diretamente com suas credenciais (sem interação do usuário final).

    O token obtido é armazenado em cache na instância e reutilizado enquanto
    válido. Quando expira, um novo token é solicitado automaticamente, sem
    interromper o fluxo do usuário.

    Atributos de classe (constantes):
        KEY    (str): Client ID da aplicação registrada no Gateway BNDES.
        SECRET (str): Client Secret correspondente ao KEY.
        URL    (str): Endpoint OAuth2 para solicitação de tokens.

    Atributos de instância:
        _token  (str | None): Token de acesso atual (Bearer Token).
        _expira (float)     : Timestamp Unix do momento de expiração do token.
    """

    # Credenciais de acesso ao Gateway de APIs do BNDES
    KEY    = "TTeb9hR_Yf3tjQxgi1bbaMmqhtYa"
    SECRET = "Xfvp9bdw2em69Q8I4Xpjy2EPUNwa"
    URL    = "https://apis-gateway.bndes.gov.br/token"

    def __init__(self):
        """Inicializa sem token. O primeiro token é obtido na primeira chamada."""
        self._token  = None   # Nenhum token disponível ao iniciar
        self._expira = 0      # Timestamp 0 = já expirado, força renovação imediata

    def _token_expirado(self) -> bool:
        """
        Verifica se o token atual está ausente ou expirado.

        Compara o timestamp atual com o momento de expiração registrado.
        Um margem de 60 segundos é subtraída na renovação para evitar uso
        de token na borda da expiração.

        Returns:
            bool: True se o token precisa ser renovado, False se ainda é válido.
        """
        return self._token is None or datetime.now().timestamp() >= self._expira

    def obter_token(self) -> str | None:
        """
        Retorna um token de acesso válido, renovando-o automaticamente se necessário.

        Este é o único método público da classe. Todas as chamadas de API
        devem usar este método para obter o token — nunca acessar _token diretamente.

        Returns:
            str | None: Bearer Token ativo, ou None se a renovação falhar.
        """
        if self._token_expirado():
            self._renovar()   # Renova silenciosamente antes de retornar
        return self._token

    def _renovar(self):
        """
        Solicita um novo token ao Gateway BNDES via HTTP POST.

        Envia as credenciais usando autenticação HTTP Basic (Base64).
        Em caso de sucesso, armazena o token e calcula o timestamp de
        expiração com margem de segurança de 60 segundos.

        Em caso de falha (timeout, erro de rede), exibe aviso e mantém
        o estado anterior sem lançar exceção, para não quebrar o fluxo.
        """
        # Autenticação Basic: codifica KEY:SECRET em Base64 no cabeçalho
        auth = requests.auth.HTTPBasicAuth(self.KEY, self.SECRET)
        try:
            res = requests.post(
                self.URL,
                data={"grant_type": "client_credentials"},  # Fluxo OAuth2 sem usuário
                auth=auth,
                verify=False,   # Ignora validação SSL (ambiente corporativo BNDES)
                timeout=10      # Abandona após 10 segundos sem resposta
            )
            dados = res.json()

            self._token = dados.get("access_token")

            # Calcula expiração: agora + duração_informada - 60s de margem de segurança
            self._expira = datetime.now().timestamp() + dados.get("expires_in", 3600) - 60

        except Exception as e:
            # Falha silenciosa: registra no console mas não interrompe o sistema
            print(f"⚠️ Erro ao renovar token: {e}")


print("✅ Bloco 1 carregado.")

✅ Bloco 1 carregado.


## 🧠 Bloco 2 — Inteligência de Negócio (Regras FNE)
*Validação de NCM via BrasilAPI e triagem de passibilidade.*

In [2]:
# =============================================================================
# BLOCO 2 — INTELIGÊNCIA DE NEGÓCIO: VALIDAÇÃO NCM E REGRAS FNE
# =============================================================================
# O NCM (Nomenclatura Comum do Mercosul) é o código de 8 dígitos que classifica
# mercadorias para fins tributários e aduaneiros. No contexto do FNE, ele é
# usado para determinar se um bem precisa de análise técnica do BNDES antes
# do financiamento.
#
# Regra de negócio aplicada:
#   - NCMs iniciados em 84 (máquinas), 87 (veículos), 82 (ferramentas) ou
#     20 (produtos alimentícios processados) exigem consulta ao Catálogo CFI.
#   - Demais NCMs são passíveis de financiamento direto, sem etapa adicional.
# =============================================================================


# =============================================================================
# CLASSE: ValidadorFNE
# =============================================================================
class ValidadorFNE:
    """
    Aplica as regras de elegibilidade do FNE sobre um código NCM.

    Responsabilidades:
        1. Verificar a existência e validade de um NCM via BrasilAPI.
        2. Determinar se o NCM exige análise técnica complementar no BNDES.

    Todos os métodos são estáticos ou de classe: não é necessário instanciar.
    Uso: ValidadorFNE.consultar_ncm('84295219')

    Atributo de classe:
        PREFIXOS_OBRIGATORIOS (tuple): Prefixos de 2 dígitos que obrigam
            consulta ao Catálogo CFI antes do financiamento.
    """

    # Prefixos NCM que, pela regra FNE, exigem validação no Catálogo CFI do BNDES
    # 84 = Máquinas e equipamentos mecânicos
    # 87 = Veículos automóveis, tratores
    # 82 = Ferramentas, artigos de cutelaria
    # 20 = Preparações de produtos hortícolas / alimentícios
    PREFIXOS_OBRIGATORIOS = ('84', '87', '82', '20')

    @staticmethod
    def consultar_ncm(ncm: str) -> dict | None:
        """
        Consulta os dados de um NCM na BrasilAPI (base oficial governamental).

        A BrasilAPI é uma API pública brasileira que agrega dados de diversas
        fontes governamentais, incluindo a tabela NCM da Receita Federal.

        Args:
            ncm (str): Código NCM com exatamente 8 dígitos numéricos.
                       Ex: '84295219'

        Returns:
            dict | None: Dicionário com os dados do NCM (inclui 'descricao',
                         'codigo', etc.) se encontrado, ou None em caso de
                         erro, timeout ou NCM inexistente.
        """
        try:
            res = requests.get(
                f"https://brasilapi.com.br/api/ncm/v1/{ncm}",
                timeout=10   # Abandona se a BrasilAPI não responder em 10s
            )
            # Retorna os dados apenas se a API confirmar que o NCM existe (200 OK)
            return res.json() if res.status_code == 200 else None
        except Exception:
            # Qualquer falha de rede retorna None sem travar o sistema
            return None

    @classmethod
    def exige_analise_bndes(cls, ncm: str) -> bool:
        """
        Determina se um NCM exige consulta obrigatória ao Catálogo CFI do BNDES.

        Esta é a implementação da regra de negócio central do sistema:
        bens com NCMs nos prefixos definidos precisam ter sua elegibilidade
        confirmada no Catálogo de Itens Financiáveis (CFI) antes do FNE.

        Args:
            ncm (str): Código NCM de 8 dígitos.

        Returns:
            bool: True = exige análise BNDES / False = financiamento direto.
        """
        # str.startswith() aceita uma tupla e verifica qualquer dos prefixos
        return ncm.startswith(cls.PREFIXOS_OBRIGATORIOS)


print("✅ Bloco 2 carregado.")

✅ Bloco 2 carregado.


## 📡 Bloco 3 — Camada de Serviços BNDES
*Integração com o endpoint de busca do Catálogo CFI.*

In [3]:
# =============================================================================
# BLOCO 3 — CAMADA DE SERVIÇOS: INTEGRAÇÃO COM O CATÁLOGO CFI DO BNDES
# =============================================================================
# O Catálogo de Itens Financiáveis (CFI) é a base oficial do BNDES que lista
# todos os equipamentos e máquinas elegíveis para financiamento.
#
# Cada item possui um Código Finame (CFI) único, que identifica:
#   - Fabricante e CNPJ
#   - Nome e modelo do produto
#   - NCM associado
#   - Situação cadastral (Financiável / Cancelado / etc.)
#
# Esta classe encapsula todas as chamadas HTTP à API do BNDES, isolando
# o restante do sistema dos detalhes de autenticação e endpoints.
# =============================================================================


# =============================================================================
# CLASSE: BNDESService
# =============================================================================
class BNDESService:
    """
    Camada de acesso ao Catálogo de Itens Financiáveis (CFI) do BNDES.

    Encapsula as requisições HTTP à API oficial, injetando o token de
    autenticação automaticamente via instância de BNDESAuth.

    Responsabilidades:
        - Montar os cabeçalhos HTTP com Bearer Token atualizado.
        - Realizar buscas no Catálogo CFI por keyword (nome, CNPJ ou CFI).
        - Fornecer o link oficial de um produto no portal BNDES.

    Atributos de classe (constantes de URL):
        BASE_URL (str): Endpoint da API de busca de itens financiáveis.
        CFI_URL  (str): URL base do portal público do Catálogo CFI.

    Args:
        auth (BNDESAuth): Instância autenticadora que fornece tokens válidos.
    """

    # Endpoint oficial de busca no Catálogo CFI (API REST do BNDES)
    BASE_URL = "https://apis-gateway.bndes.gov.br/catalogoCFI/v1/itemfinanciavel/buscar"

    # URL base do portal web do Catálogo CFI (usado para gerar links clicáveis)
    CFI_URL  = "https://ws.bndes.gov.br/cfi_catalogo/produto"

    def __init__(self, auth: BNDESAuth):
        """Recebe a instância de autenticação para uso interno nas requisições."""
        self.auth = auth

    def _headers(self) -> dict:
        """
        Constrói os cabeçalhos HTTP necessários para as requisições autenticadas.

        Chama obter_token() a cada requisição, garantindo que o token
        esteja sempre atualizado (a renovação automática ocorre internamente).

        Returns:
            dict: Cabeçalhos com Authorization (Bearer) e Accept (JSON).
        """
        return {
            "Authorization": f"Bearer {self.auth.obter_token()}",  # Token renovado automaticamente
            "Accept": "application/json"                           # Solicita resposta em JSON
        }

    def buscar_itens(self, keyword: str, quantidade: int = 30) -> list:
        """
        Realiza busca universal no Catálogo CFI usando o parâmetro 'keyword'.

        O parâmetro keyword é flexível: aceita nome do produto, CNPJ do
        fabricante ou Código CFI (Finame). A API retorna os itens mais
        relevantes para o termo informado.

        Args:
            keyword    (str): Termo de busca (nome, CNPJ ou código CFI).
            quantidade (int): Número máximo de resultados. Padrão: 30.

        Returns:
            list: Lista de dicionários com os itens financiáveis encontrados.
                  Cada item contém: codigoFiname, fabricante, cnpjFabricante,
                  nomeItem, modeloItem, numeroNcm, posicaoCadastral.
                  Retorna lista vazia em caso de falha ou sem resultados.
        """
        try:
            res = requests.get(
                self.BASE_URL,
                params={"keyword": keyword, "quantidade": quantidade},
                headers=self._headers(),   # Injeta token de autenticação
                verify=False,              # SSL desabilitado (rede corporativa)
                timeout=15                 # Timeout maior: API BNDES pode ser lenta
            )
            # A API retorna os itens dentro da chave 'entidades'
            return res.json().get("entidades", []) if res.status_code == 200 else []
        except Exception:
            return []   # Falha silenciosa: retorna lista vazia

    @classmethod
    def link_oficial(cls, cfi: str) -> str:
        """
        Gera o link público do produto no portal web do Catálogo CFI.

        Este link é usado tanto no rodapé do PDF (como hyperlink clicável)
        quanto no screenshot da página web que é anexado ao relatório.

        Args:
            cfi (str): Código CFI (Finame) do produto. Ex: '03447782'

        Returns:
            str: URL completa da página do produto no portal BNDES.
        """
        return f"{cls.CFI_URL}/{cfi}"


print("✅ Bloco 3 carregado.")

✅ Bloco 3 carregado.


## 📄 Bloco 4 — Gerador de Relatório PDF
*Dossiê formal com cabeçalho institucional, tabela de dados e rodapé.*

In [4]:
# =============================================================================
# BLOCO 4 — GERAÇÃO DE RELATÓRIO PDF E CAPTURA DE PÁGINA WEB
# =============================================================================
# Este bloco é composto por três camadas:
#
#   1. FUNÇÕES UTILITÁRIAS: formatação de CNPJ e geração de timestamp
#
#   2. capturar_pagina_bndes(): abre o Chrome em modo headless (invisível),
#      acessa a URL do produto no site do BNDES, tira um screenshot de toda
#      a página e divide em fatias de proporção A4 para inserção no PDF.
#
#   3. GeradorPDF: classe principal que monta o PDF em duas partes:
#      - Página 1: dossiê formal com cabeçalho, tabela de dados e situação FNE
#      - Páginas 2+: screenshot fatiado do site oficial do produto no BNDES
# =============================================================================


# =============================================================================
# FUNÇÕES UTILITÁRIAS
# =============================================================================

def formatar_cnpj(cnpj: str) -> str:
    """
    Formata um CNPJ numérico bruto para o padrão visual brasileiro.

    Remove qualquer caractere não numérico, garante 14 dígitos com
    zeros à esquerda e aplica a máscara XX.XXX.XXX/XXXX-XX.

    Args:
        cnpj (str): CNPJ em qualquer formato (só números, com pontuação, etc.).
                    Ex: '02833372000124' ou '02.833.372/0001-24'

    Returns:
        str: CNPJ formatado. Ex: '02.833.372/0001-24'
    """
    c = re.sub(r'\D', '', str(cnpj)).zfill(14)   # Remove não-dígitos e garante 14 chars
    return f"{c[:2]}.{c[2:5]}.{c[5:8]}/{c[8:12]}-{c[12:]}"


def log_brasilia() -> str:
    """
    Retorna a data e hora atual no fuso horário de Brasília (GMT-3).

    Usado para registrar o momento exato de cada consulta e geração de PDF,
    independentemente do fuso do servidor onde o Colab está executando.

    Returns:
        str: Data e hora formatadas. Ex: '17/03/2026 08:43:06'
    """
    return datetime.now(pytz.timezone("America/Sao_Paulo")).strftime("%d/%m/%Y %H:%M:%S")


# =============================================================================
# FUNÇÃO: capturar_pagina_bndes
# =============================================================================

def capturar_pagina_bndes(url: str) -> list:
    """
    Captura um screenshot completo de uma página web e a divide em fatias A4.

    Processo detalhado:
        1. Detecta automaticamente os caminhos do Chromium e ChromeDriver
           instalados no ambiente (evita falhas por caminho fixo incorreto).
        2. Inicia o Chrome em modo headless (sem janela visível) no Colab.
        3. Acessa a URL informada e aguarda 4 segundos para o JS renderizar.
        4. Expande a janela do navegador para a altura total do conteúdo,
           garantindo que toda a página seja capturada (não só a parte visível).
        5. Salva o screenshot completo em /tmp/bndes_full.png.
        6. Abre a imagem com Pillow e calcula a altura equivalente a uma
           página A4 na proporção da largura capturada (ratio 210:297).
        7. Faz um loop de cortes (crop) da imagem, de cima para baixo,
           cada um do tamanho de uma página A4.
        8. A última fatia (menor que A4) recebe um fundo branco para
           completar o tamanho e evitar distorções no PDF.
        9. Cada fatia é salva em /tmp/bndes_pag_N.png e seu caminho
           é adicionado à lista de retorno.
       10. O driver é encerrado no bloco 'finally' para liberar memória,
           mesmo se ocorrer um erro durante a captura.

    Args:
        url (str): URL completa da página a capturar.
                   Ex: 'https://ws.bndes.gov.br/cfi_catalogo/produto/03447782'

    Returns:
        list[str]: Lista de caminhos de arquivos PNG temporários, um por
                   página A4. Ex: ['/tmp/bndes_pag_0.png', '/tmp/bndes_pag_1.png']
    """
    print("📸 Capturando página BNDES via Playwright...")

    # ── Garante que o Chromium do Playwright está disponível ─────────────────
    # Se o kernel foi reiniciado após a instalação, o browser pode não existir.
    # Este bloco verifica e baixa automaticamente se necessário.
    import subprocess, sys
    from playwright._impl._driver import compute_driver_executable
    try:
        # Tenta importar o caminho do executável — lança exceção se não existir
        from playwright.sync_api import sync_playwright as _sp
        with _sp() as _pw:
            _ = _pw.chromium.executable_path  # Apenas acessa o atributo
    except Exception:
        pass  # Segue para o download abaixo se necessário
    # Roda 'playwright install chromium' silenciosamente como fallback
    result = subprocess.run(
        [sys.executable, '-m', 'playwright', 'install', 'chromium', '--with-deps'],
        capture_output=True, text=True
    )
    if result.returncode != 0 and 'already' not in result.stdout:
        print(f"   playwright install: {result.stdout.strip() or result.stderr.strip()}")

    caminho_full = "/tmp/bndes_full.png"

    # ── Coroutine assíncrona interna ──────────────────────────────────────────
    # Definida como async porque o Colab roda em loop asyncio e a API síncrona
    # do Playwright não funciona nesse contexto. nest_asyncio permite que
    # asyncio.run() seja chamado de dentro do loop já existente.
    async def _capturar():
        async with async_playwright() as pw:
            # Inicia o Chromium próprio do Playwright (isolado do sistema)
            browser = await pw.chromium.launch(
                headless=True,
                args=['--no-sandbox', '--disable-dev-shm-usage']
            )
            # Cria página com viewport inicial
            page = await browser.new_page(viewport={'width': 1240, 'height': 900})

            # Navega e aguarda rede ociosa (JS e recursos carregados)
            await page.goto(url, wait_until='networkidle', timeout=30000)

            # Obtém dimensões reais da página renderizada
            largura = await page.evaluate("document.body.scrollWidth")
            altura  = await page.evaluate("document.body.scrollHeight")
            print(f"   página   : {largura}x{altura}px")

            # Ajusta viewport para capturar toda a altura da página
            await page.set_viewport_size({'width': max(largura, 1240), 'height': altura})

            # full_page=True garante captura integral sem scroll
            await page.screenshot(path=caminho_full, full_page=True)
            print(f"   screenshot salvo em {caminho_full}")

            await browser.close()

    # asyncio.run() executa a coroutine; nest_asyncio permite isso no Colab
    asyncio.run(_capturar())

    # ── Divisão da imagem em fatias A4 ───────────────────────────────────────
    img  = Image.open(caminho_full)
    w, h = img.size
    print(f"   imagem   : {w}x{h}px")

    # Calcula a altura A4 na proporção da largura capturada
    # Proporção A4: largura 210mm × altura 297mm → ratio = 297/210
    h_a4   = int(w * 297 / 210)
    fatias = []
    topo   = 0
    idx    = 0

    while topo < h:
        base  = min(topo + h_a4, h)              # Limite inferior da fatia atual
        fatia = img.crop((0, topo, w, base))     # Recorta a fatia da imagem original

        # Última fatia: pode ser menor que A4 → preenche com fundo branco
        if base - topo < h_a4:
            fundo = Image.new('RGB', (w, h_a4), (255, 255, 255))  # Fundo branco A4
            fundo.paste(fatia, (0, 0))   # Cola a fatia no topo do fundo branco
            fatia = fundo

        caminho = f"/tmp/bndes_pag_{idx}.png"
        fatia.save(caminho, 'PNG')
        fatias.append(caminho)
        topo += h_a4    # Avança para a próxima fatia
        idx  += 1

    print(f"✅ Página capturada em {len(fatias)} parte(s).")
    return fatias


# =============================================================================
# CLASSE: GeradorPDF
# =============================================================================

class GeradorPDF:
    """
    Gera o Dossiê de Elegibilidade FNE em formato PDF com layout formal.

    O relatório é composto por:
        - Página 1: Dossiê formal
            • Cabeçalho institucional com faixa azul BNB
            • Metadados da consulta (data, hora, NCM)
            • Tabela de dados técnicos com linhas alternadas
            • Bloco de situação no FNE em destaque verde
            • Rodapé com link clicável para o portal BNDES
        - Páginas 2+: Anexo web
            • Screenshot fatiado da página oficial do produto no BNDES
            • Cada fatia equivale a uma página A4
            • Cabeçalho discreto em cada página de anexo

    Paleta de cores (RGB) — identidade visual BNB/BNDES:
        AZUL_BNB   : Azul institucional do BNB  → (0, 67, 136)
        AZUL_CLARO : Azul claro para tabela      → (240, 247, 255)
        VERDE      : Verde para situação positiva → (39, 174, 96)
        CINZA      : Cinza para textos secundários → (100, 100, 100)
        BRANCO     : Branco puro                  → (255, 255, 255)

    Args:
        item (dict): Dicionário com os dados do item financiável (retornado pelo BNDES).
        ncm  (str) : Código NCM consultado (8 dígitos).
        log  (str) : Timestamp da consulta formatado (ex: '17/03/2026 08:43:06').
    """

    # Paleta de cores RGB usada em todo o documento
    AZUL_BNB   = (0, 67, 136)     # Azul institucional BNB
    AZUL_CLARO = (240, 247, 255)  # Fundo das linhas pares da tabela
    VERDE      = (39, 174, 96)    # Bloco de situação (financiável)
    CINZA      = (100, 100, 100)  # Textos de metadados e rodapé
    BRANCO     = (255, 255, 255)  # Texto sobre fundos coloridos

    def __init__(self, item: dict, ncm: str, log: str):
        """Inicializa o gerador com os dados do item e configura o documento FPDF."""
        self.item = item
        self.ncm  = ncm
        self.log  = log
        self.pdf  = FPDF()
        # auto_page_break=False: garante que todo o conteúdo fique na página 1
        # sem criar uma segunda página por overflow de conteúdo
        self.pdf.set_auto_page_break(auto=False)

    def _cabecalho(self):
        """
        Renderiza o cabeçalho institucional no topo da página.

        Composto por:
            - Faixa azul BNB (rect preenchido) de 0 a 28mm de altura
            - Título principal em branco, negrito, tamanho 18
            - Subtítulo institucional em branco, tamanho 10
            - Linha horizontal fina azul abaixo da faixa
        """
        p = self.pdf
        # Faixa azul de fundo do cabeçalho (largura total A4 = 210mm)
        p.set_fill_color(*self.AZUL_BNB)
        p.rect(0, 0, 210, 28, style='F')   # style='F' = preenchido sem borda

        # Título em branco sobre a faixa azul
        p.set_font("Helvetica", "B", 18)
        p.set_text_color(*self.BRANCO)
        p.set_xy(10, 6)
        p.cell(190, 10, "DOSSIE DE ELEGIBILIDADE FNE", align='C')

        # Subtítulo institucional
        p.set_font("Helvetica", "", 10)
        p.set_xy(10, 16)
        p.cell(190, 8, "Banco do Nordeste do Brasil  |  Analise de Itens Financiaveis", align='C')

        # Linha divisória fina abaixo da faixa
        p.set_draw_color(*self.AZUL_BNB)
        p.set_line_width(0.5)
        p.line(10, 30, 200, 30)   # Linha de x=10 a x=200, na altura y=30mm
        p.ln(10)

    def _metadados(self):
        """
        Renderiza a linha de metadados da consulta abaixo do cabeçalho.

        Exibe à esquerda: data/hora da consulta em Brasília.
        Exibe à direita: NCM consultado.
        Fonte pequena (9pt) em cinza para não competir com o conteúdo principal.
        """
        p = self.pdf
        p.set_xy(10, 35)   # Posição fixa abaixo do cabeçalho
        p.set_font("Helvetica", "", 9)
        p.set_text_color(*self.CINZA)
        p.cell(95, 6, f"Data/Hora: {self.log} (Brasilia)", align='L')   # Metadado esquerdo
        p.cell(95, 6, f"NCM Consultado: {self.ncm}", align='R')          # Metadado direito
        p.ln(10)

    def _titulo_secao(self, texto: str):
        """
        Renderiza um título de seção com fundo azul e texto branco.

        Funciona como um separador visual entre blocos de conteúdo.
        O texto é recuado 2mm à esquerda para melhor alinhamento visual.

        Args:
            texto (str): Texto do título da seção. Ex: 'DADOS TECNICOS DO EQUIPAMENTO'
        """
        p = self.pdf
        p.ln(4)   # Espaço antes do título
        p.set_fill_color(*self.AZUL_BNB)
        p.set_text_color(*self.BRANCO)
        p.set_font("Helvetica", "B", 10)
        p.cell(190, 8, f"  {texto}", fill=True, new_x=XPos.LMARGIN, new_y=YPos.NEXT)   # fill=True = fundo colorido
        p.ln(2)   # Espaço após o título

    def _tabela_dados(self, linhas: list):
        """
        Renderiza a tabela de dados técnicos com linhas alternadas.

        Cada linha da tabela é composta por duas células:
            - Célula de rótulo (55mm): negrito, fundo alternado
            - Célula de valor (135mm): regular, mesmo fundo alternado

        As linhas pares recebem fundo AZUL_CLARO e as ímpares fundo BRANCO,
        criando o efeito zebrado que facilita a leitura.

        Args:
            linhas (list): Lista de tuplas (label, valor).
                           Ex: [('Fabricante', 'JCB DO BRASIL LTDA'), ...]
        """
        p    = self.pdf
        cores = [self.AZUL_CLARO, self.BRANCO]   # Alterna entre par e ímpar
        for idx, (label, valor) in enumerate(linhas):
            p.set_fill_color(*cores[idx % 2])    # idx % 2: 0=par=azul, 1=ímpar=branco
            p.set_text_color(60, 60, 60)          # Cinza escuro para melhor legibilidade
            p.set_font("Helvetica", "B", 10)
            p.cell(55, 9, f"  {label}:", fill=True, border=1)                     # Rótulo
            p.set_font("Helvetica", "", 10)
            p.cell(135, 9, f"  {str(valor or 'N/D')}", fill=True, border=1, new_x=XPos.LMARGIN, new_y=YPos.NEXT)  # Valor

    def _situacao(self):
        """
        Renderiza o bloco de conclusão sobre a situação do bem no FNE.

        Exibe uma faixa verde com texto branco contendo a sentença formal
        de elegibilidade, no formato:
            'O Bem {produto} codigo {cfi} e passivel de financiamento pelo FNE'

        Usa multi_cell (em vez de cell) para suportar nomes longos de produtos
        que possam quebrar em mais de uma linha automaticamente.
        """
        p       = self.pdf
        produto = self.item.get("nomeItem", "N/D")
        cfi     = self.item.get("codigoFiname", "N/D")
        # Nota: fpdf2 não suporta acentos sem fonte Unicode — texto sem acento intencional
        texto = f"O Bem {produto} codigo {cfi} e passivel de financiamento pelo FNE"
        p.ln(6)
        p.set_fill_color(*self.VERDE)
        p.set_text_color(*self.BRANCO)
        p.set_font("Helvetica", "B", 11)
        p.multi_cell(190, 10, texto, fill=True, align='C')   # multi_cell para quebra automática

    def _rodape(self):
        """
        Renderiza o rodapé da página 1 com link clicável para o portal BNDES.

        Posicionado a -28mm do final da página (absoluto), garantindo que
        fique sempre na parte inferior independentemente do conteúdo acima.

        Composto por:
            - Linha horizontal fina cinza como separador visual
            - Texto de assinatura automática em itálico cinza
            - URL do produto sublinhada, em azul e clicável no PDF
              (usando o parâmetro 'link' da célula fpdf2)
        """
        p    = self.pdf
        cfi  = self.item.get('codigoFiname', '')
        link = BNDESService.link_oficial(cfi)   # Monta a URL do portal BNDES

        p.set_y(-28)   # Posição absoluta: 28mm antes do final da página
        p.set_draw_color(*self.CINZA)
        p.line(10, p.get_y(), 200, p.get_y())   # Linha separadora horizontal
        p.ln(3)

        # Linha 1: texto informativo em itálico
        p.set_font("Helvetica", "I", 8)
        p.set_text_color(*self.CINZA)
        p.cell(190, 5, "Dossie gerado automaticamente pelo sistema FNE v6.1.", align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)

        # Linha 2: link clicável — 'IU' = itálico + sublinhado; link= torna clicável no PDF
        p.set_font("Helvetica", "IU", 8)
        p.set_text_color(0, 67, 136)   # Azul BNB para o link
        p.cell(190, 5, link, align='C', link=link)

    def _paginas_web(self):
        """
        Adiciona páginas de anexo com screenshot do site oficial do BNDES.

        Para cada fatia A4 retornada por capturar_pagina_bndes():
            1. Cria uma nova página no PDF
            2. Adiciona cabeçalho discreto (faixa azul fina, 10mm) com
               identificação CFI e número da parte
            3. Insere a imagem da fatia ocupando o espaço restante (287mm)
            4. Remove o arquivo PNG temporário para liberar espaço em disco

        Em caso de qualquer falha (sem internet, site fora, timeout),
        exibe um aviso mas não interrompe a geração do PDF principal.
        """
        cfi  = self.item.get('codigoFiname', '')
        link = BNDESService.link_oficial(cfi)
        try:
            fatias = capturar_pagina_bndes(link)   # Obtém lista de imagens fatiadas
            for idx, caminho in enumerate(fatias):
                self.pdf.add_page()   # Nova página A4 para cada fatia

                # Cabeçalho discreto: faixa azul fina de 10mm no topo
                self.pdf.set_fill_color(*self.AZUL_BNB)
                self.pdf.rect(0, 0, 210, 10, style='F')
                self.pdf.set_font("Helvetica", "I", 7)
                self.pdf.set_text_color(*self.BRANCO)
                self.pdf.set_xy(10, 2)
                self.pdf.cell(
                    190, 6,
                    f"Anexo - Pagina oficial BNDES (CFI {cfi})  |  Parte {idx + 1} de {len(fatias)}",
                    align='C'
                )

                # Imagem da fatia web: começa em y=10mm, ocupa os 287mm restantes
                self.pdf.image(caminho, x=0, y=10, w=210, h=287)

                os.remove(caminho)   # Limpa arquivo temporário após inserção

        except Exception as e:
            # Falha tolerada: PDF continua sendo gerado sem o anexo
            print(f"⚠️ Não foi possível capturar a página web: {e}")
            print("   O PDF será gerado sem o anexo do site BNDES.")

    def gerar(self) -> str:
        """
        Orquestra a montagem completa do PDF e salva o arquivo.

        Sequência de construção:
            1. Cria a Página 1 (dossiê): cabeçalho → metadados → título de seção
               → tabela de dados → bloco de situação → rodapé
            2. Chama _paginas_web() para adicionar as páginas de anexo (2+)
            3. Salva o PDF com nome baseado no código CFI

        Returns:
            str: Nome do arquivo PDF gerado. Ex: 'Relatorio_FNE_03447782.pdf'
        """
        i = self.item
        p = self.pdf

        # ── Página 1: Dossiê Formal ───────────────────────────────────────────
        p.add_page()
        self._cabecalho()                              # Faixa azul + título
        self._metadados()                              # Data, hora e NCM
        self._titulo_secao("DADOS TECNICOS DO EQUIPAMENTO")  # Barra azul de seção
        self._tabela_dados([
            ("Fabricante",  i.get("fabricante")),
            ("CNPJ",        formatar_cnpj(i.get("cnpjFabricante", ""))),
            ("Produto",     i.get("nomeItem")),
            ("Modelo",      i.get("modeloItem")),
            ("Codigo CFI",  i.get("codigoFiname")),
            ("NCM",         i.get("numeroNcm")),
        ])
        self._situacao()                               # Bloco verde de conclusão
        self._rodape()                                 # Rodapé com link clicável

        # ── Páginas 2+: Anexo Web (screenshot do site BNDES) ─────────────────
        self._paginas_web()

        # Salva o arquivo com nome baseado no código CFI do produto
        nome = f"Relatorio_FNE_{i.get('codigoFiname', 'sem_cfi')}.pdf"
        p.output(nome)
        return nome


print("✅ Bloco 4 carregado.")

✅ Bloco 4 carregado.


## 📋 Bloco 5 — Controle de Versão e CHANGELOG
*Registro histórico de versões do sistema e log de relatórios gerados na sessão.*

In [5]:
# =============================================================================
# BLOCO 5 — CONTROLE DE VERSÃO E CHANGELOG
# =============================================================================
# Este bloco implementa o sistema de versionamento do projeto FNE.
#
# Duas camadas de registro:
#
#   1. CHANGELOG (estático): histórico permanente de alterações do código,
#      mantido manualmente pelo desenvolvedor a cada nova versão.
#      Cada entrada registra: versão, data, autor, tipo e descrição.
#
#   2. LOG_SESSAO (dinâmico): registro automático de todos os PDFs gerados
#      durante a sessão atual do Colab. É preenchido pelo sistema a cada
#      geração e zerado quando o runtime é reiniciado.
#
# Tipos de alteração no CHANGELOG:
#   'major'   → Nova funcionalidade principal ou refatoração significativa
#   'feature' → Funcionalidade menor ou melhoria incremental
#   'fix'     → Correção de bug ou ajuste de comportamento
#
# Para adicionar uma nova versão:
#   1. Incremente VERSAO_ATUAL
#   2. Adicione uma nova entrada no início da lista CHANGELOG
#   3. Documente o tipo: 'major', 'feature' ou 'fix'
# =============================================================================


# Versão atual do sistema — incrementar a cada alteração significativa
VERSAO_ATUAL = "6.1"

# =============================================================================
# CHANGELOG — Histórico permanente de alterações
# Lista ordenada do mais recente para o mais antigo.
# =============================================================================
CHANGELOG = [
    {
        "versao": "6.1",
        "data": "17/03/2026",
        "autor": "Fábio",
        "tipo": "feature",
        "descricao": "Versionamento: CHANGELOG fixo + log automático de PDFs gerados na sessão."
    },
    {
        "versao": "6.0",
        "data": "17/03/2026",
        "autor": "Fábio",
        "tipo": "major",
        "descricao": "Anexo web: screenshot automático do site BNDES adicionado ao PDF."
    },
    {
        "versao": "6.0",
        "data": "17/03/2026",
        "autor": "Fábio",
        "tipo": "fix",
        "descricao": "PDF: página única, texto de situação em linguagem natural, link clicável no rodapé."
    },
    {
        "versao": "6.0",
        "data": "17/03/2026",
        "autor": "Fábio",
        "tipo": "major",
        "descricao": "Reescrita completa: aba de Comparação, token automático, PDF reformulado, código modular."
    },
    {
        "versao": "5.0",
        "data": "Anterior",
        "autor": "Fábio",
        "tipo": "major",
        "descricao": "Versão original: validação NCM, busca CFI, ficha técnica, PDF básico."
    },
]

# =============================================================================
# LOG_SESSAO — Registro automático de PDFs gerados na sessão atual
# Lista vazia ao iniciar; preenchida por registrar_pdf_gerado() a cada PDF.
# É zerada quando o runtime do Colab é reiniciado.
# =============================================================================
LOG_SESSAO = []
# Estrutura de cada entrada:
# {
#   'timestamp': '17/03/2026 08:43:06',  # Momento exato da geração
#   'versao':    '6.1',                  # Versão do sistema que gerou
#   'produto':   'ESCAVADEIRA JS220LC',  # Nome do bem financiável
#   'cfi':       '03447782',             # Código CFI do produto
#   'ncm':       '84295219',             # NCM consultado
#   'arquivo':   'Relatorio_FNE_....pdf' # Nome do arquivo gerado
# }


def registrar_pdf_gerado(item: dict, ncm: str, arquivo: str):
    """
    Registra no LOG_SESSAO os dados de um PDF gerado durante a sessão.

    Chamada automaticamente pelo DashboardFNE._gerar_pdf() após cada
    geração bem-sucedida. Não precisa ser chamada manualmente.

    Args:
        item    (dict): Dicionário do item financiável (retornado pela API BNDES).
        ncm     (str) : Código NCM consultado.
        arquivo (str) : Nome do arquivo PDF gerado.
    """
    LOG_SESSAO.append({
        "timestamp": log_brasilia(),                  # Horário exato de Brasília
        "versao":    VERSAO_ATUAL,                    # Rastreia qual versão gerou
        "produto":   item.get("nomeItem", "N/D"),
        "cfi":       item.get("codigoFiname", "N/D"),
        "ncm":       ncm,
        "arquivo":   arquivo,
    })


def exportar_changelog():
    """
    Exporta o CHANGELOG e o log da sessão para um arquivo JSON.

    Gera um JSON estruturado com:
        - Identificação do sistema e versão
        - Timestamp da exportação
        - Lista completa do CHANGELOG histórico
        - Lista de todos os PDFs gerados na sessão atual

    O arquivo é salvo localmente no Colab e o download é iniciado
    automaticamente para o computador do usuário.

    Arquivo gerado: CHANGELOG_FNE_v{VERSAO_ATUAL}.json
    """
    dados = {
        "sistema":      f"Dossiê de Inteligência FNE v{VERSAO_ATUAL}",
        "exportado_em": log_brasilia(),
        "changelog":    CHANGELOG,    # Histórico completo de versões
        "log_sessao":   LOG_SESSAO,   # PDFs gerados nesta sessão
    }
    nome = f"CHANGELOG_FNE_v{VERSAO_ATUAL}.json"
    with open(nome, 'w', encoding='utf-8') as f:
        json.dump(dados, f, ensure_ascii=False, indent=2)  # indent=2 para legibilidade
    files.download(nome)
    print(f"✅ '{nome}' exportado com sucesso.")


def html_changelog() -> str:
    """
    Gera o HTML da tabela de histórico de versões para exibição no painel.

    Cada entrada do CHANGELOG é renderizada como uma linha da tabela.
    O tipo da alteração recebe um badge colorido para identificação rápida:
        - major   → badge azul escuro
        - feature → badge verde
        - fix     → badge laranja

    Returns:
        str: String HTML completa com a tabela de versões.
    """
    # Mapeamento tipo → (cor do texto, cor de fundo do badge)
    cores_tipo = {
        "major":   ("#004388", "#e8f0fb"),
        "feature": ("#27ae60", "#eafaf1"),
        "fix":     ("#e67e22", "#fff8e1"),
    }
    linhas = ""
    for e in CHANGELOG:
        cor_txt, cor_bg = cores_tipo.get(e['tipo'], ("#555", "#f9f9f9"))
        # Badge colorido com borda no estilo do tipo de alteração
        badge = (
            f"<span style='background:{cor_bg};color:{cor_txt};"
            f"border:1px solid {cor_txt};border-radius:4px;"
            f"padding:2px 7px;font-size:0.78em;font-weight:bold;'>"
            f"{e['tipo'].upper()}</span>"
        )
        linhas += (
            f"<tr style='border-bottom:1px solid #eee;'>"
            f"<td style='padding:8px;font-weight:bold;color:#004388;white-space:nowrap;'>v{e['versao']}</td>"
            f"<td style='padding:8px;color:#888;white-space:nowrap;'>{e['data']}</td>"
            f"<td style='padding:8px;'>{badge}</td>"
            f"<td style='padding:8px;color:#333;'>{e['descricao']}</td>"
            "</tr>"
        )
    return (
        "<table style='width:100%;border-collapse:collapse;font-family:Arial;font-size:0.9em;'>"
        "<thead><tr style='background:#004388;color:#fff;'>"
        "<th style='padding:8px 10px;text-align:left;'>Versão</th>"
        "<th style='padding:8px 10px;text-align:left;'>Data</th>"
        "<th style='padding:8px 10px;text-align:left;'>Tipo</th>"
        "<th style='padding:8px 10px;text-align:left;'>Descrição</th>"
        "</tr></thead>"
        f"<tbody>{linhas}</tbody></table>"
    )


def html_log_sessao() -> str:
    """
    Gera o HTML da tabela de PDFs gerados na sessão atual.

    Os registros são exibidos do mais recente para o mais antigo
    (usando reversed()). Linhas alternadas recebem fundo levemente azulado.
    Se nenhum PDF foi gerado ainda, exibe mensagem orientativa.

    Returns:
        str: String HTML com a tabela de log ou mensagem de estado vazio.
    """
    if not LOG_SESSAO:
        return (
            "<div style='padding:20px;text-align:center;color:#aaa;font-family:Arial;'>"
            "Nenhum relatório gerado nesta sessão ainda.</div>"
        )
    linhas = ""
    for idx, e in enumerate(reversed(LOG_SESSAO)):   # Mais recente no topo
        bg = "#f8f9ff" if idx % 2 == 0 else "#ffffff"   # Zebrado azulado
        linhas += (
            f"<tr style='background:{bg};border-bottom:1px solid #eee;'>"
            f"<td style='padding:7px 10px;color:#888;white-space:nowrap;'>{e['timestamp']}</td>"
            f"<td style='padding:7px 10px;font-weight:bold;color:#004388;'>{e['produto']}</td>"
            f"<td style='padding:7px 10px;'>{e['cfi']}</td>"
            f"<td style='padding:7px 10px;'>{e['ncm']}</td>"
            f"<td style='padding:7px 10px;color:#27ae60;'>{e['arquivo']}</td>"
            "</tr>"
        )
    return (
        "<table style='width:100%;border-collapse:collapse;font-family:Arial;font-size:0.88em;'>"
        "<thead><tr style='background:#002a5c;color:#fff;'>"
        "<th style='padding:8px 10px;text-align:left;'>Data/Hora</th>"
        "<th style='padding:8px 10px;text-align:left;'>Produto</th>"
        "<th style='padding:8px 10px;text-align:left;'>CFI</th>"
        "<th style='padding:8px 10px;text-align:left;'>NCM</th>"
        "<th style='padding:8px 10px;text-align:left;'>Arquivo</th>"
        "</tr></thead>"
        f"<tbody>{linhas}</tbody></table>"
    )


print(f"✅ Bloco 5 carregado — Sistema FNE v{VERSAO_ATUAL}")

✅ Bloco 5 carregado — Sistema FNE v6.1


## 💻 Bloco 6 — Painel Analítico FNE v6.1
*Interface com abas: Consulta, Comparação e Versões.*

In [6]:
# =============================================================================
# BLOCO 6 — PAINEL ANALÍTICO FNE v6.1 (INTERFACE INTERATIVA)
# =============================================================================
# Este bloco monta toda a interface visual do sistema usando ipywidgets.
#
# Arquitetura da interface:
#   ┌─────────────────────────────────────────────────────┐
#   │  Cabeçalho gradiente azul (título + versão)         │
#   ├──────────────┬──────────────┬───────────────────────┤
#   │ 🔍 Consulta │ 📊 Comparação│ 📋 Versões            │
#   │              │              │                       │
#   │ NCM → Busca  │ Tabela lado  │ CHANGELOG + Log       │
#   │ → Dropdown   │ a lado de    │ da sessão             │
#   │ → Ficha      │ até 4 itens  │ + Exportar JSON       │
#   │ → PDF/Comp.  │              │                       │
#   └──────────────┴──────────────┴───────────────────────┘
#
# Padrão de eventos ipywidgets:
#   - Botões: .on_click(handler)      → handler recebe o evento de clique
#   - Dropdown: .observe(handler)     → handler recebe {'new': valor_selecionado}
#   - Output: widgets.Output()        → contexto 'with' captura print/display
#
# Separação de responsabilidades na classe DashboardFNE:
#   _build_*   → Constroem os widgets e layout (chamados no __init__)
#   _validar   → Handler do botão 'Validar NCM'
#   _pesquisar → Handler do botão 'Pesquisar'
#   _detalhar  → Handler do dropdown de resultados
#   _gerar_pdf → Handler do botão 'Baixar PDF'
#   _adicionar/limpar/renderizar_comparacao → Gerenciam a aba Comparação
#   _atualizar_aba_versoes → Atualiza a aba Versões após cada PDF
#   exibir     → Renderiza o painel completo na célula
# =============================================================================


# =============================================================================
# HELPERS DE HTML — Funções de apoio para renderização de fichas e tabelas
# =============================================================================

def _linha_tabela(label: str, valor) -> str:
    """
    Gera o HTML de uma linha (<tr>) para a tabela da ficha técnica.

    Cada linha é composta por duas células:
        - Célula de rótulo (28% da largura): negrito, cinza escuro
        - Célula de valor (restante): regular, quase preto
    Ambas têm borda inferior fina como separador visual entre linhas.

    Args:
        label (str): Texto do rótulo. Ex: 'Fabricante'
        valor      : Valor a exibir. Se None ou vazio, exibe '—'.

    Returns:
        str: String HTML de uma linha <tr> completa.
    """
    return (
        "<tr>"
        f"<td style='padding:6px 8px;width:28%;color:#555;font-weight:bold;"
        f"border-bottom:1px solid #f0f0f0;'>{label}</td>"
        f"<td style='padding:6px 8px;border-bottom:1px solid #f0f0f0;color:#222;'>{valor or '—'}</td>"
        "</tr>"
    )


def html_ficha(item: dict) -> str:
    """
    Gera o HTML completo da ficha técnica de um item financiável.

    Exibida na área de output da Aba 1 ao selecionar um item no dropdown.
    Composta por:
        - Cabeçalho com título e timestamp da consulta
        - Tabela de atributos usando _linha_tabela()
        - O campo 'Código CFI' é renderizado como link clicável para o portal BNDES
        - Bloco verde com a situação cadastral do item no FNE
        - Rodapé com fonte dos dados

    Args:
        item (dict): Dicionário do item retornado pela API BNDES.

    Returns:
        str: String HTML completa da ficha técnica.
    """
    i    = item
    cfi  = i.get("codigoFiname", "")
    link = BNDESService.link_oficial(cfi)   # Link clicável para o portal BNDES

    # Monta todas as linhas da tabela de atributos
    rows = (
        _linha_tabela("Fabricante", i.get("fabricante")) +
        _linha_tabela("CNPJ", formatar_cnpj(i.get("cnpjFabricante", ""))) +
        _linha_tabela("Produto", i.get("nomeItem")) +
        _linha_tabela("Modelo", i.get("modeloItem")) +
        # CFI como hyperlink azul clicável (target='_blank' = nova aba)
        _linha_tabela("Código CFI",
            f"<a href='{link}' target='_blank' style='color:#004388;font-weight:bold;'>{cfi}</a>") +
        _linha_tabela("NCM", i.get("numeroNcm"))
    )
    situacao = i.get("posicaoCadastral", "N/D")

    # Monta o card completo da ficha técnica
    return (
        "<div style='border:1.5px solid #004388;padding:20px;border-radius:14px;"
        "background:#fff;box-shadow:0 4px 16px rgba(0,67,136,0.10);margin-top:10px;'>"
        # Linha superior: título + timestamp
        "<div style='display:flex;justify-content:space-between;align-items:center;'>"
        "<h3 style='color:#004388;margin:0;font-family:Arial;'>📄 Ficha Técnica</h3>"
        f"<span style='font-size:0.8em;color:#888;'>🕒 {log_brasilia()} (Brasília)</span>"
        "</div>"
        "<hr style='border:0.5px solid #e0e8f4;margin:12px 0;'>"
        # Tabela de atributos
        f"<table style='width:100%;border-collapse:collapse;font-family:Arial;font-size:0.95em;'>{rows}</table>"
        # Bloco de situação no FNE com borda verde
        "<div style='margin-top:14px;padding:10px 16px;border-radius:8px;"
        "background:#eafaf1;border-left:5px solid #27ae60;'>"
        f"<b style='color:#27ae60;font-size:1.05em;'>✅ Situação no FNE: {situacao}</b>"
        "</div>"
        # Rodapé com fonte dos dados
        "<p style='font-size:0.75em;color:#aaa;text-align:center;margin-top:10px;'>"
        "Dados extraídos via API Oficial BNDES · Catálogo CFI</p>"
        "</div>"
    )


# =============================================================================
# ESTILOS CSS DAS ABAS
# =============================================================================

def injetar_estilos():
    """
    Injeta CSS global para estilização das abas e botões do painel.

    Os seletores .lm-TabBar-tab e .p-TabBar-tab cobrem as duas versões
    do JupyterLab/Colab (prefixo 'lm-' nas versões mais recentes,
    'p-' nas mais antigas).

    Efeitos aplicados:
        - Abas inativas: fundo azul claro, bordas arredondadas
        - Aba ativa: fundo azul BNB, texto branco, negrito
        - Conteúdo das abas: cantos arredondados inferiores, sombra suave
        - Botões: cantos arredondados, hover com leve transparência,
          active com escala reduzida (feedback visual de clique)
    """
    display(HTML("""<style>
        /* Estilo das abas inativas */
        .lm-TabBar-tab, .p-TabBar-tab {
            background:#f0f4fb !important;
            border-radius:12px 12px 0 0 !important;
            margin-right:5px !important;
            border:1px solid #d0dce8 !important;
            font-family:Arial,sans-serif;
        }
        /* Estilo da aba ativa (selecionada) */
        .lm-TabBar-tab.lm-mod-current, .p-TabBar-tab.p-mod-current {
            background:#004388 !important;
            color:white !important;
            font-weight:bold !important;
        }
        /* Área de conteúdo abaixo das abas */
        .widget-tab-contents {
            border-radius:0 0 14px 14px !important;
            box-shadow:0 8px 32px rgba(0,67,136,0.10) !important;
            background:#fff !important;
            padding:18px !important;
        }
        /* Estilo base dos botões */
        .jupyter-button {
            border-radius:8px !important;
            font-weight:600 !important;
            font-family:Arial,sans-serif !important;
        }
        .jupyter-button:hover  { opacity:0.88 !important; }           /* Hover suave */
        .jupyter-button:active { transform:scale(0.96) !important; }  /* Feedback de clique */
    </style>"""))


# =============================================================================
# CLASSE: DashboardFNE
# =============================================================================

class DashboardFNE:
    """
    Painel analítico interativo do sistema FNE v6.1.

    Gerencia o estado da aplicação e coordena todas as interações do usuário
    com as APIs (BNDES e BrasilAPI) e com o gerador de PDF.

    Estado interno mantido:
        auth        (BNDESAuth)   : Instância de autenticação (token compartilhado)
        service     (BNDESService): Instância de acesso à API CFI
        resultados  (list)        : Itens retornados pela última busca
        item_atual  (dict | None) : Item selecionado no dropdown
        comparacao  (list)        : Itens adicionados à aba de comparação (máx. 4)
        out_main    (Output)      : Área de saída da Aba 1 (ficha, erros, logs)
        out_comp    (Output)      : Área de saída da Aba 2 (tabela comparativa)
        out_versoes (Output)      : Área de saída da Aba 3 (changelog e log)

    Fluxo principal de uso:
        1. Usuário digita NCM → clica 'Validar NCM' → _validar()
        2. Se NCM exige análise → digita termo → clica 'Pesquisar' → _pesquisar()
        3. Seleciona item no dropdown → _detalhar() exibe a ficha
        4. Clica 'Baixar PDF' → _gerar_pdf() gera e baixa o relatório
           OU clica 'Comparar' → _adicionar_comparacao() adiciona à Aba 2
    """

    def __init__(self):
        """
        Inicializa o painel: injeta estilos, cria instâncias de serviço,
        inicializa o estado e constrói todos os widgets e abas.
        """
        injetar_estilos()   # Aplica CSS global antes de criar os widgets

        # Instâncias de serviço compartilhadas entre todos os handlers
        self.auth       = BNDESAuth()             # Gerencia tokens OAuth2
        self.service    = BNDESService(self.auth) # Acessa API do Catálogo CFI

        # Estado da aplicação
        self.resultados = []    # Itens da última busca
        self.item_atual = None  # Item atualmente selecionado
        self.comparacao = []    # Fila de itens para comparação (máx. 4)

        # Áreas de output para cada aba (contexto 'with' para capturar saídas)
        self.out_main   = widgets.Output()
        self.out_comp   = widgets.Output()

        # Constrói cada aba na sequência correta
        self._build_consulta()    # Aba 1: formulário de consulta
        self._build_comparacao()  # Aba 2: tabela comparativa
        self._build_tabs()        # Junta tudo + constrói Aba 3 (versões)

    # =========================================================================
    # MÉTODOS DE CONSTRUÇÃO (_build_*)
    # Responsáveis por criar widgets, definir layouts e vincular eventos.
    # Chamados apenas uma vez no __init__.
    # =========================================================================

    def _build_consulta(self):
        """
        Constrói todos os widgets da Aba 1 — Consulta de Elegibilidade.

        Widgets criados:
            ncm_in  : Campo de texto para o NCM (8 dígitos)
            btn_val : Botão 'Validar NCM' — inicia o fluxo de consulta
            bus_in  : Campo de texto para a busca (desabilitado até NCM válido)
            btn_bus : Botão 'Pesquisar' — aciona busca na API BNDES
            drp     : Dropdown com os resultados da busca
            btn_pdf : Botão 'Baixar PDF' (desabilitado até item selecionado)
            btn_add : Botão 'Comparar' (desabilitado até item selecionado)

        Layout: VBox vertical com HBoxes horizontais para agrupar
        campo + botão na mesma linha.
        """
        self.ncm_in  = widgets.Text(
            placeholder='8 dígitos', description='NCM:',
            layout=widgets.Layout(width='260px'))
        self.btn_val = widgets.Button(
            description='1. Validar NCM', button_style='primary',
            layout=widgets.Layout(width='145px'))
        self.bus_in  = widgets.Text(
            placeholder='Nome, CNPJ ou CFI', description='Busca:',
            disabled=True, layout=widgets.Layout(width='340px'))
        self.btn_bus = widgets.Button(
            description='2. Pesquisar', button_style='success',
            disabled=True, layout=widgets.Layout(width='130px'))
        self.drp = widgets.Dropdown(
            description='Resultado:',
            options=[('— aguardando busca —', -1)],
            disabled=True,
            layout=widgets.Layout(width='98%'))
        self.btn_pdf = widgets.Button(
            description='📄 Baixar PDF', button_style='danger',
            disabled=True, layout=widgets.Layout(width='160px'))
        self.btn_add = widgets.Button(
            description='➕ Comparar', button_style='warning',
            disabled=True, layout=widgets.Layout(width='130px'))

        self.btn_val.on_click(self._validar)
        self.btn_bus.on_click(self._pesquisar)
        self.drp.observe(self._detalhar, names='value')
        self.btn_pdf.on_click(self._gerar_pdf)
        self.btn_add.on_click(self._adicionar_comparacao)

        self.aba_consulta = widgets.VBox([
            widgets.HTML("<h4 style='color:#004388;font-family:Arial;'>🔍 Consulta de Elegibilidade</h4>"),
            widgets.HBox([self.ncm_in,  self.btn_val]),
            widgets.HBox([self.bus_in,  self.btn_bus]),
            self.drp,
            widgets.HBox([self.btn_pdf, self.btn_add]),
            self.out_main
        ])

    # ── Aba 2 ─────────────────────────────────────────────────────────────────

    def _build_comparacao(self):
        """
        Constrói os widgets da Aba 2 — Comparação de Itens.

        A aba é composta por:
            - Texto explicativo sobre como adicionar itens
            - Botão 'Limpar' para esvaziar a lista de comparação
            - out_comp: área de output onde a tabela comparativa é renderizada

        A tabela em si é gerada dinamicamente por _renderizar_comparacao()
        sempre que um item é adicionado ou a lista é limpa.
        """
        self.btn_limpar = widgets.Button(
            description='🗑️ Limpar', button_style='',
            layout=widgets.Layout(width='120px'))
        self.btn_limpar.on_click(self._limpar_comparacao)
        self.aba_comparacao = widgets.VBox([
            widgets.HTML(
                "<h4 style='color:#004388;font-family:Arial;'>📊 Comparação de Itens</h4>"
                "<p style='color:#666;font-size:0.9em;font-family:Arial;'>"
                "Adicione itens pela aba <b>Consulta</b> usando o botão "
                "<em>➕ Comparar</em>. Até <b>4 itens</b> simultâneos.</p>"),
            self.btn_limpar,
            self.out_comp
        ])

    def _build_versoes(self):
        self.btn_exportar = widgets.Button(
            description='⬇️ Exportar CHANGELOG.json',
            button_style='',
            layout=widgets.Layout(width='240px')
        )
        self.btn_exportar.on_click(lambda _: exportar_changelog())
        self.out_versoes = widgets.Output()
        self.aba_versoes = widgets.VBox([
            widgets.HTML(
                f"<h4 style='color:#004388;font-family:Arial;'>📋 Controle de Versão</h4>"
                f"<p style='color:#666;font-size:0.88em;font-family:Arial;'>"
                f"Versão atual: <b>v{VERSAO_ATUAL}</b></p>"
            ),
            self.btn_exportar,
            self.out_versoes,
        ])
        with self.out_versoes:
            display(HTML(
                "<h5 style='font-family:Arial;color:#333;margin-top:16px;'>Histórico de Versões</h5>"
            ))
            display(HTML(html_changelog()))
            display(HTML(
                "<h5 style='font-family:Arial;color:#333;margin-top:20px;'>Log desta Sessão</h5>"
            ))
            display(HTML(html_log_sessao()))


    def _build_tabs(self):
        """
        Constrói a Aba 3 (Versões) e monta o widget Tab com as três abas.

        A Aba 3 é construída aqui (e não em _build_versoes separado) pois
        precisa do out_versoes para ser inicializado antes de popular
        o changelog pela primeira vez.

        As três abas são:
            0: '🔍  Consulta FNE'  → aba_consulta
            1: '📊  Comparação'    → aba_comparacao
            2: '📋  Versões'       → aba_versoes
        """
        self._build_versoes()   # Constrói a Aba 3 antes de registrar nos tabs
        self.tabs = widgets.Tab(children=[self.aba_consulta, self.aba_comparacao, self.aba_versoes])
        self.tabs.set_title(0, '🔍  Consulta FNE')
        self.tabs.set_title(1, '📊  Comparação')
        self.tabs.set_title(2, '📋  Versões')

    # =========================================================================
    # HANDLERS DE EVENTO
    # Cada método corresponde a uma ação do usuário na interface.
    # Todos recebem '_' como argumento (evento ignorado — só o estado importa).
    # =========================================================================

    def _validar(self, _):
        """
        Handler do botão '1. Validar NCM'.

        Fluxo:
            1. Limpa a área de output anterior
            2. Extrai apenas os dígitos do campo NCM
            3. Valida que tem exatamente 8 dígitos
            4. Consulta a BrasilAPI para verificar existência do NCM
            5. Se válido: exibe descrição em card azul
               - Se exige análise BNDES: desbloqueia campo de busca (laranja)
               - Se financiamento direto: exibe mensagem final verde
        """
        ncm = re.sub(r'\D', '', self.ncm_in.value)
        with self.out_main:
            if len(ncm) != 8:
                display(HTML("<b style='color:red;'>❌ Informe exatamente 8 dígitos numéricos.</b>"))
                return
            dados = ValidadorFNE.consultar_ncm(ncm)
            if not dados:
                display(HTML("<b style='color:red;'>❌ NCM não encontrado na base governamental.</b>"))
                return
            descricao = (dados.get('descricao') or '')[:160]
            display(HTML(
                "<div style='background:#f0f7ff;padding:12px 16px;border-radius:10px;"
                "border-left:5px solid #004388;font-family:Arial;'>"
                f"<b>✅ NCM {ncm} validado:</b><br>"
                f"<span style='color:#333;'>{descricao}...</span></div>"
            ))
            if ValidadorFNE.exige_analise_bndes(ncm):
                self.bus_in.disabled  = False
                self.btn_bus.disabled = False
                display(HTML(
                    "<div style='margin-top:8px;padding:10px 14px;border-radius:8px;"
                    "background:#fff8e1;border-left:5px solid #f39c12;font-family:Arial;'>"
                    "<b style='color:#e67e22;'>⚠️ NCM exige análise técnica BNDES. "
                    "Consulta ao Catálogo CFI liberada.</b></div>"
                ))
            else:
                display(HTML(
                    f"<div style='margin-top:8px;padding:12px 16px;border-radius:10px;"
                    "background:#eafaf1;border-left:5px solid #27ae60;font-family:Arial;'>"
                    f"<b style='color:#27ae60;font-size:1.1em;'>"
                    f"🏁 NCM {ncm} é passível de financiamento direto.</b></div>"
                ))

    def _pesquisar(self, _):
        """
        Handler do botão '2. Pesquisar'.

        Fluxo:
            1. Valida que o campo de busca não está vazio
            2. Exibe mensagem de carregamento com timestamp
            3. Chama BNDESService.buscar_itens() com o keyword informado
            4. Se encontrar resultados: popula o dropdown com opções no
               formato 'CFI: XXXXX | Fabricante | Produto'
            5. Se não encontrar: exibe mensagem informativa
        """
        self.out_main.clear_output()
        keyword = self.bus_in.value.strip()
        if not keyword:
            with self.out_main:
                display(HTML("<b style='color:red;'>❌ Informe um termo para busca.</b>"))
            return
        with self.out_main:
            print(f"📡 Consultando API BNDES... [{log_brasilia()}]")
        self.resultados = self.service.buscar_itens(keyword)
        self.out_main.clear_output()
        with self.out_main:
            if not self.resultados:
                display(HTML("<b style='color:#888;'>ℹ️ Nenhum item encontrado.</b>"))
                return
            self.drp.options = [('— Selecione um item —', -1)] + [
                (f"CFI: {i.get('codigoFiname')}  |  {i.get('fabricante')}  |  {i.get('nomeItem')}", idx)
                for idx, i in enumerate(self.resultados)
            ]
            self.drp.disabled = False
            self.drp.value    = -1
            total = len(self.resultados)
            display(HTML(
                f"<b style='color:#004388;'>✅ {total} item(ns) encontrado(s). Selecione no menu acima.</b>"
            ))

    def _detalhar(self, change):
        """
        Handler do dropdown de resultados (observe 'value').

        Chamado automaticamente sempre que o usuário muda a seleção.
        Se o valor selecionado for -1 (placeholder), desabilita os botões.
        Se for um índice válido, carrega o item correspondente de self.resultados,
        habilita os botões de PDF e Comparar, e exibe a ficha técnica.

        Args:
            change (dict): Dicionário de mudança do observe.
                           change['new'] = valor atual do dropdown.
        """
        if change['new'] == -1:
            self.btn_pdf.disabled = True
            self.btn_add.disabled = True
            return
        self.item_atual = self.resultados[change['new']]
        self.btn_pdf.disabled = False
        self.btn_add.disabled = False
        self.out_main.clear_output(wait=True)
        with self.out_main:
            display(HTML(html_ficha(self.item_atual)))

    def _gerar_pdf(self, _):
        """
        Handler do botão '📄 Baixar PDF'.

        Fluxo:
            1. Verifica se há item selecionado (segurança)
            2. Extrai o NCM do campo de texto
            3. Instancia GeradorPDF e chama .gerar() → retorna nome do arquivo
            4. Registra no LOG_SESSAO via registrar_pdf_gerado()
            5. Atualiza a Aba 3 com o novo registro no log
            6. Inicia o download do arquivo para o computador do usuário
        """
        if not self.item_atual:   # Segurança: botão pode ter sido clicado sem item
            return
        with self.out_main:
            print("⏳ Gerando PDF...")
            ncm  = re.sub(r'\D', '', self.ncm_in.value)
            nome = GeradorPDF(self.item_atual, ncm, log_brasilia()).gerar()
            registrar_pdf_gerado(self.item_atual, ncm, nome)
            self._atualizar_aba_versoes()
            print(f"✅ '{nome}' gerado! Iniciando download...")
            files.download(nome)

    def _adicionar_comparacao(self, _):
        """
        Handler do botão '➕ Comparar'.

        Validações antes de adicionar:
            - Existe item selecionado?
            - A lista já tem 4 itens (limite máximo)?
            - O item (pelo CFI) já está na lista (evita duplicatas)?

        Se passar em todas: adiciona à self.comparacao, chama
        _renderizar_comparacao() para atualizar a Aba 2 e exibe
        confirmação verde com contador (X/4).
        """
        if not self.item_atual:   # Segurança: botão pode ter sido clicado sem item
            return
        with self.out_main:
            if len(self.comparacao) >= 4:
                display(HTML("<b style='color:orange;'>⚠️ Limite de 4 itens atingido.</b>"))
                return
            if self.item_atual.get('codigoFiname') in [i.get('codigoFiname') for i in self.comparacao]:
                display(HTML("<b style='color:orange;'>⚠️ Item já está na comparação.</b>"))
                return
            self.comparacao.append(self.item_atual)
            self._renderizar_comparacao()
            display(HTML(
                f"<div style='margin-top:8px;padding:8px 14px;background:#eafaf1;"
                f"border-radius:8px;font-family:Arial;'>"
                f"<b style='color:#27ae60;'>✅ Adicionado à comparação ({len(self.comparacao)}/4).</b></div>"
            ))

    def _limpar_comparacao(self, _):
        """
        Handler do botão '🗑️ Limpar'.

        Esvazia completamente a lista self.comparacao e chama
        _renderizar_comparacao() para atualizar a Aba 2 com a
        mensagem de estado vazio.
        """
        self.comparacao = []          # Esvazia a lista de itens comparados
        self._renderizar_comparacao() # Atualiza a Aba 2 com estado vazio

    def _renderizar_comparacao(self):
        """
        Renderiza a tabela de comparação na Aba 2.

        Chamada sempre que self.comparacao muda (adição ou limpeza).
        Se a lista estiver vazia, exibe mensagem orientativa.

        Estrutura da tabela gerada:
            - Cabeçalho: coluna 'Atributo' (azul escuro) + uma coluna
              por item adicionado (azul BNB), com CFI e fabricante
            - Linhas alternadas (zebrado azulado) para cada atributo:
              Fabricante, CNPJ, Produto, Modelo, Código CFI, NCM, Situação FNE
            - CNPJ formatado via formatar_cnpj()
            - Situação FNE: verde se 'ATIVO', vermelho caso contrário
            - Rodapé com contagem de itens e fonte dos dados
        """
        self.out_comp.clear_output(wait=True)  # wait=True evita flickering
        with self.out_comp:
            if not self.comparacao:
                display(HTML(
                    "<div style='padding:30px;text-align:center;color:#aaa;font-family:Arial;'>"
                    "Nenhum item adicionado.<br>"
                    "Use <b>➕ Comparar</b> na aba Consulta.</div>"
                ))
                return

            campos = [
                ('Fabricante',   'fabricante'),
                ('CNPJ',         'cnpjFabricante'),
                ('Produto',      'nomeItem'),
                ('Modelo',       'modeloItem'),
                ('Código CFI',   'codigoFiname'),
                ('NCM',          'numeroNcm'),
                ('Situação FNE', 'posicaoCadastral'),
            ]

            headers = "".join([
                "<th style='background:#004388;color:#fff;padding:10px 8px;"
                "text-align:center;font-family:Arial;font-size:0.9em;'>"
                f"CFI {i.get('codigoFiname','')}<br>"
                f"<small style='font-weight:normal;'>{i.get('fabricante','')}</small></th>"
                for i in self.comparacao
            ])

            linhas_html = ""
            for idx, (label, campo) in enumerate(campos):
                bg = "#f0f4fb" if idx % 2 == 0 else "#ffffff"
                celulas = ""
                for item in self.comparacao:
                    val = item.get(campo) or '—'
                    if campo == 'cnpjFabricante':
                        val = formatar_cnpj(val)
                    if campo == 'posicaoCadastral':
                        cor = '#27ae60' if 'ATIVO' in str(val).upper() else '#e74c3c'
                        val = f"<b style='color:{cor};'>{val}</b>"
                    celulas += (
                        f"<td style='padding:8px;text-align:center;font-family:Arial;"
                        f"font-size:0.88em;border-bottom:1px solid #e8edf5;'>{val}</td>"
                    )
                linhas_html += (
                    f"<tr style='background:{bg};'>"
                    f"<td style='padding:8px 10px;font-weight:bold;font-family:Arial;"
                    f"font-size:0.88em;color:#555;border-bottom:1px solid #e8edf5;'>{label}</td>"
                    f"{celulas}</tr>"
                )

            display(HTML(
                "<div style='overflow-x:auto;'>"
                "<table style='width:100%;border-collapse:collapse;"
                "border:1.5px solid #d0dce8;'>"
                "<thead><tr>"
                "<th style='background:#002a5c;color:#fff;padding:10px 8px;"
                "font-family:Arial;font-size:0.9em;'>Atributo</th>"
                f"{headers}</tr></thead>"
                f"<tbody>{linhas_html}</tbody></table>"
                f"<p style='font-size:0.75em;color:#aaa;text-align:center;"
                f"margin-top:10px;font-family:Arial;'>"
                f"{len(self.comparacao)} item(ns) · API Oficial BNDES</p></div>"
            ))

    def _atualizar_aba_versoes(self):
        """
        Recarrega o conteúdo da Aba 3 — Versões após cada PDF gerado.

        Limpa a área de output e re-renderiza:
            1. Título 'Histórico de Versões' + tabela do CHANGELOG
            2. Título 'Log desta Sessão' + tabela do LOG_SESSAO atualizado

        Usa wait=True no clear_output para evitar flickering visual
        (a limpeza só ocorre quando o novo conteúdo estiver pronto).
        """
        self.out_versoes.clear_output(wait=True)  # Limpa antes de re-renderizar
        with self.out_versoes:
            display(HTML(
                "<h5 style='font-family:Arial;color:#333;margin-top:16px;'>Histórico de Versões</h5>"
            ))
            display(HTML(html_changelog()))
            display(HTML(
                "<h5 style='font-family:Arial;color:#333;margin-top:20px;'>Log desta Sessão</h5>"
            ))
            display(HTML(html_log_sessao()))

    # ── Exibir ────────────────────────────────────────────────────────────────

    def exibir(self):
        """
        Renderiza o painel completo na célula do Colab.

        Monta um VBox com dois elementos:
            1. Cabeçalho gradiente azul com título e subtítulo
            2. O widget Tab com as três abas

        Este é o único método público do Dashboard e o ponto de entrada
        da interface. Deve ser chamado uma única vez na última célula.
        """
        display(widgets.VBox([
            widgets.HTML(
                "<div style='padding:16px 20px;background:linear-gradient(135deg,#004388,#0066cc);"
                "border-radius:14px;margin-bottom:10px;'>"
                "<h2 style='color:#fff;margin:0;font-family:Arial;'>"
                "🏦 Painel Analítico FNE <span style='font-weight:300;'>v6.1</span></h2>"
                "<p style='color:#cde;margin:4px 0 0;font-family:Arial;font-size:0.88em;'>"
                "Auditoria e Elegibilidade · Jornada CODERS</p></div>"
            ),
            self.tabs,
        ]))


# =============================================================================
# INICIALIZAÇÃO DO PAINEL
# =============================================================================
# Instancia o DashboardFNE e chama exibir().
# O __init__ executa toda a construção de widgets internamente.
# exibir() renderiza o painel na saída desta célula.
#
# Nota: reexecutar esta célula reinicia o painel (estado zerado),
# mas NÃO reinicia o token OAuth2 nem o LOG_SESSAO (mantidos nos
# objetos globais auth e LOG_SESSAO do Bloco 5).
# =============================================================================
app = DashboardFNE()
app.exibir()